# M0 pilot evaluation on a free GPU (Kaggle / Colab)

Runs `scripts/run_pilot_eval.py` (retrieval + our detector vs baseline B1, Hindi, synthetic
pilot set) on a CUDA GPU. On the project's CPU laptop the NLI step takes many hours; on a T4
it takes minutes (DECISIONS D-026). The command and arguments are **identical** to the local
run, so the numbers are comparable; only floating-point rounding differs between devices.

**Kaggle:** *Create → New Notebook → File → Import Notebook* (upload this file) →
*Settings → Accelerator: GPU T4* and *Internet: on* → *Run All*. Download
`m0_pilot_results.zip` from the *Output* panel when it finishes.

**Colab:** *File → Upload notebook* → *Runtime → Change runtime type → T4 GPU* → *Run all*.
Download `m0_pilot_results.zip` from the file browser.

No tokens are needed: the repository is public and every model and dataset used here is
ungated. Nothing is written back to GitHub; the zip is copied into the repo by hand and
committed with the rest of the M0 results.

In [ ]:
import os
import pathlib

# Which code to run. The pilot script lives on this branch until bundle 4 is merged.
BRANCH = "feat/m0-pilot"
REPO_URL = "https://github.com/YousufFFFF/indic-rag-hallu.git"
# Same arguments as the local CPU run (see results/m0/pilot.json -> run config).
PILOT_ARGS = "--embedder me5-small --k 10 --context-passages 3 --require-clean"

WORK = pathlib.Path("/kaggle/working" if pathlib.Path("/kaggle/working").exists() else "/content")
REPO = WORK / "indic-rag-hallu"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
print("working dir:", WORK)

In [ ]:
# 1. A GPU must be attached, or this would be as slow as the laptop.
import sys

import torch

print("python", sys.version.split()[0], "| torch", torch.__version__)
assert torch.cuda.is_available(), "No GPU: turn on a GPU accelerator in the notebook settings."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Fresh clone of the branch (the run refuses a dirty tree, and records the commit).
!rm -rf {REPO}
!git clone --quiet --depth 1 --branch {BRANCH} {REPO_URL} {REPO}
!git -C {REPO} log -1 --format='commit %H  %s'

In [ ]:
# 3. Dependencies at the repo's pinned versions (pyproject.toml). The platform's own CUDA
#    build of torch and its numpy are kept; their versions are recorded in the run's meta.json.
!pip install -q transformers==5.23.0 sentence-transformers==6.3.0 faiss-cpu==1.16.0 \
    indic-nlp-library==0.92 pydantic==2.14.4 pyyaml==6.0.3 scikit-learn==1.9.1
# The package itself; requires-python pins 3.11 for Docker, the code also runs on newer 3.x.
!pip install -q --no-deps --ignore-requires-python -e {REPO}

In [ ]:
# 4. Hindi IndicQA, pinned revision and verified hashes (about 3 MB).
%cd {REPO}
!python scripts/download_indicqa.py --lang hi

In [ ]:
# 5. The pilot run: retrieval -> pilot set -> ours and B1 -> thresholds tuned on val -> test.
!git status --porcelain
!python -u scripts/run_pilot_eval.py {PILOT_ARGS}

In [ ]:
# 6. Check the run and package everything the repo needs into one zip.
import json
import zipfile

summary = json.loads((REPO / "results/m0/pilot.json").read_text("utf-8"))
run_dir = REPO / "results/runs" / summary["run_id"]
meta = json.loads((run_dir / "meta.json").read_text("utf-8"))
print("run:", summary["run_id"], "| commit:", summary["git_commit"], "| dirty:", summary["dirty"])
print("environment:", json.dumps(meta.get("environment", {}), indent=1)[:800])
for name, res in summary["detection_test"].items():
    print(
        f"{name:16s} response F1 {res['response']['pooled']['f1']}  "
        f"span F1 {res['spans']['pooled']['span_f1']}  thresholds {res['thresholds']}"
    )

out = WORK / "m0_pilot_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(REPO / "results/m0/pilot.json", "results/m0/pilot.json")
    for f in sorted(run_dir.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(REPO).as_posix())
print("wrote", out, f"({out.stat().st_size / 1e6:.1f} MB)")